## Context

talking back to the client while server is running ..

In [11]:
from fastmcp import FastMCP, Context, Client
import asyncio

ctxserver = FastMCP("cts-demo")

@ctxserver.tool
async def send_mails(emails:list[str], context:Context) -> str:
    """Send Emails to many users at a time"""
    total = len(emails)
    for i in range(total):
        email = emails[i]
        await context.report_progress(progress=i+1, total=total,message=f"Sending mail to : -> {email}")
        
        await asyncio.sleep(2)
    return "Email send."

In [12]:
async def watch_email_process(progress, total, message):
    print(message)
    
async with Client(ctxserver) as client:
    res = await client.call_tool("send_mails", 
                                 {"emails":["example@gmail.com", "fastmcp@gmail.com"]}, 
                                 progress_handler=watch_email_process
                                 )
    print(res)
    

Sending mail to : -> example@gmail.com
Sending mail to : -> fastmcp@gmail.com
CallToolResult(content=[TextContent(type='text', text='Email send.', annotations=None, meta=None)], structured_content={'result': 'Email send.'}, meta={'fastmcp': {'wrap_result': True}, 'io.modelcontextprotocol/serverInfo': {'name': 'cts-demo', 'version': '4.2.6'}}, data='Email send.', is_error=False)


# Long Running Task

In [ ]:
from fastmcp import FastMCP, Client
from fastmcp.dependencies import Progress
from fastmcp.utilities.tasks import TaskConfig
from fastmcp_tasks import TasksExtension, call_tool_task

mcp = FastMCP("Task Server")
mcp.add_extension(TasksExtension())

@mcp.tool(task=TaskConfig(mode='required'))
async def slow_compitation(email_counts:int, progress: Progress = Progress()) -> str:
    """A long Running task / operation..."""
    await progress.set_total(email_counts)
    for i in range(email_counts):
        # Sending the email code
        await asyncio.sleep(1)
        await progress.set_message(f'sending mail {i}/{email_counts}')
        await progress.increment()
        
    return f"Completed in {email_counts} seconds.."
    

In [47]:
async with Client(mcp) as client:
    task = await call_tool_task(client,"slow_compitation", {"email_counts":10})
    print("Task ID",task.task_id)
    
    while True:
        status = await task.status()
        print('Status: ', status.status, status.status_message)
       
        if status.status == "completed":
            result = await task.result()
             
            print(result)
            break
        await asyncio.sleep(2)

Task ID tFtopVksACRP7fAOruZUoezA-dvU3g8PpnZ6Cv2EZng
Status:  working None
Status:  working sending mail 1/10
Status:  working sending mail 3/10
Status:  working sending mail 5/10
Status:  working sending mail 7/10
Status:  completed None
CallToolResult(content=[TextContent(type='text', text='Completed in 10 seconds..', annotations=None, meta=None)], structured_content={'result': 'Completed in 10 seconds..'}, meta={'fastmcp': {'wrap_result': True}}, data='Completed in 10 seconds..', is_error=False)
